<a href="https://colab.research.google.com/github/somyama4753l-lang/Kaggle-Datasets/blob/main/Customer%20Churn%20Modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [90]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
df_path="/content/Churn_Modelling.csv"
df=pd.read_csv(df_path)
df


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [91]:
df=df.drop(['Surname','RowNumber','CustomerId'],axis=1)
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer


In [92]:
encoded_df=pd.get_dummies(df,columns=['Geography','Gender'],drop_first=True)

In [93]:
x=encoded_df.drop('Exited',axis=1)
y=encoded_df['Exited']

In [94]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.3,shuffle=True)

In [95]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7000 entries, 5517 to 2217
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CreditScore        7000 non-null   int64  
 1   Age                7000 non-null   int64  
 2   Tenure             7000 non-null   int64  
 3   Balance            7000 non-null   float64
 4   NumOfProducts      7000 non-null   int64  
 5   HasCrCard          7000 non-null   int64  
 6   IsActiveMember     7000 non-null   int64  
 7   EstimatedSalary    7000 non-null   float64
 8   Geography_Germany  7000 non-null   bool   
 9   Geography_Spain    7000 non-null   bool   
 10  Gender_Male        7000 non-null   bool   
dtypes: bool(3), float64(2), int64(6)
memory usage: 512.7 KB


In [96]:
X_train = X_train.apply(pd.to_numeric, errors='coerce').astype(float)
X_test  = X_test.apply(pd.to_numeric, errors='coerce').astype(float)

X_train = X_train.fillna(0)
X_test  = X_test.fillna(0)

In [97]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

In [98]:
X_train_tensor=torch.tensor(X_train,dtype=torch.float32)
X_test_tensor=torch.tensor(X_test,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.to_numpy(),dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.to_numpy(),dtype=torch.float32)

In [99]:
from torch.utils.data import Dataset,DataLoader
class mydata(Dataset):
  def __init__(self,features,labels):
    self.features=features
    self.labels=labels
  def __len__(self):
    return len(self.features)
  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [100]:
train_dataset=mydata(X_train_tensor,y_train_tensor)
test_dataset=mydata(X_test_tensor,y_test_tensor)
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=True)

In [101]:
class mynn(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.layer1=nn.Linear(num_features,5)
    self.relu=nn.ReLU()
    self.layer2=nn.Linear(5,1)
    self.sigmoid=nn.Sigmoid()
  def forward(self,features):
    x=self.layer1(features)
    x=self.relu(x)
    x=self.layer2(x)
    return x



In [102]:
mymodel=mynn(X_train_tensor.shape[1])
loss=nn.BCEWithLogitsLoss()
optimizer=torch.optim.SGD(mymodel.parameters(),lr=0.001)

In [105]:
for epoch in range(50):
  for batch_x,batch_y in train_loader:
    y_pred=mymodel(batch_x)
    lossf=loss(y_pred,batch_y.view(-1,1))
    optimizer.zero_grad()
    lossf.backward()
    optimizer.step()

  print(lossf.item())



0.48764678835868835
0.5706437230110168
0.5829455852508545
0.5256705284118652
0.3913848102092743
0.3994358479976654
0.39572417736053467
0.240286722779274
0.3100353479385376
0.37318599224090576
0.2982492744922638
0.44845351576805115
0.4711218774318695
0.3746400773525238
0.32378098368644714
0.7424504160881042
0.35804930329322815
0.4890834391117096
0.6234710812568665
0.5499569773674011
0.3266795873641968
0.6430382132530212
0.264228880405426
0.5540181994438171
0.6192052364349365
0.7219789028167725
0.45290324091911316
0.4482842683792114
0.30437004566192627
0.6650012135505676
0.37935295701026917
0.43616625666618347
0.41054868698120117
0.3923024833202362
0.4792971611022949
0.2780288755893707
0.4635348618030548
0.4112456142902374
0.413100004196167
0.34041619300842285
0.3316885530948639
0.20896263420581818
0.45101913809776306
0.3224346339702606
0.4245249032974243
0.4415997564792633
0.3659447431564331
0.6674704551696777
0.40850695967674255
0.2890925407409668


In [104]:
for batch_x,batch_y in test_loader:
   y_pred=mymodel(batch_x)
   lossf=loss(y_pred,batch_y.view(-1,1))
   print(lossf.item())





0.435215026140213
0.5525463223457336
0.8219056129455566
0.3552449941635132
0.6345113515853882
0.44400477409362793
0.4114012122154236
0.36882737278938293
0.47394248843193054
0.3944627642631531
0.3315971791744232
0.7829942107200623
0.4581279456615448
0.4342744052410126
0.716845691204071
0.41471707820892334
0.2909616231918335
0.5195295810699463
0.29426324367523193
0.348945677280426
0.31440550088882446
0.3233579993247986
0.46365711092948914
0.34604310989379883
0.43788933753967285
0.358120322227478
0.3539247512817383
0.47925856709480286
0.49905115365982056
0.39120736718177795
0.3077869117259979
0.31116220355033875
0.4440542161464691
0.5014694333076477
0.4299827218055725
0.35300031304359436
0.4821030795574188
0.35120904445648193
0.5258221626281738
0.5650486946105957
0.37702712416648865
0.39757823944091797
0.5924357175827026
0.3567252457141876
0.2818220257759094
0.502715528011322
0.5596209168434143
0.5583900809288025
0.29328665137290955
0.30007100105285645
0.43588030338287354
0.43203514814376